In [1]:
import json, time
import numpy as np
import pandas as pd
import scanpy as sc
import rapids_singlecell as rsc
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import KNeighborsTransformer
from scipy.stats import spearmanr
from scipy.optimize import linear_sum_assignment
import cupy as cp
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
rmm.reinitialize(pool_allocator=True, managed_memory=True)
cp.cuda.set_allocator(rmm_cupy_allocator)

with open("benchmark_config.json") as f:
    config = json.load(f)
gcfg = config["global"]
dcfg = config["datasets"]["pbmc3k"]
prefix = dcfg["pipeline_prefix"]

SEEDS = [0, 1, 42]
cluster_results = {}

for seed in SEEDS:
    print(f"\n--- Seed {seed} ---")
    t0 = time.time()
    
    adata = sc.read_h5ad(dcfg["canonical_h5ad"])
    adata.X = adata.layers["counts"].copy()
    
    # HVG on CPU counts layer first (same as your pipeline scripts)
    sc.pp.highly_variable_genes(adata, layer="counts", n_top_genes=dcfg["n_top_genes"],
                                flavor="seurat_v3", subset=False, inplace=True)
    
    # Now move to GPU
    rsc.get.anndata_to_GPU(adata)
    rsc.pp.normalize_total(adata, target_sum=gcfg["target_sum"])
    rsc.pp.log1p(adata)
    rsc.pp.pca(adata, n_comps=gcfg["pca_n_comps"], mask_var="highly_variable")
    rsc.pp.neighbors(adata, n_neighbors=dcfg["n_neighbors"], n_pcs=dcfg["neighbors_n_pcs"],
                     use_rep="X_pca", algorithm="brute", metric=gcfg["neighbor_metric"],
                     method=gcfg["neighbor_method"], random_state=seed)
    rsc.tl.leiden(adata, resolution=dcfg["leiden_resolution"], random_state=seed, key_added="leiden")
    rsc.get.anndata_to_CPU(adata)
    
    clusters = pd.Series(adata.obs["leiden"].astype(str).values,
                         index=adata.obs_names.str.replace("-1$", "", regex=True))
    cluster_results[seed] = clusters
    
    n_cl = clusters.nunique()
    print(f"  {n_cl} clusters, {time.time()-t0:.1f}s")
    
    # Save
    pd.DataFrame({"barcode": adata.obs_names.astype(str),
                   "leiden": adata.obs["leiden"].astype(str).values}
    ).to_csv(f"write/{prefix}_rsc_gpu_015_seed{seed}_clusters.csv", index=False)
    del adata

# Compare all pairs
print(f"\n{'='*50}")
print("Seed robustness: pairwise ARI")
print(f"{'='*50}")
from itertools import combinations
for sa, sb in combinations(SEEDS, 2):
    ca, cb = cluster_results[sa], cluster_results[sb]
    common = ca.index.intersection(cb.index)
    ari = adjusted_rand_score(ca.loc[common], cb.loc[common])
    identical = (ca.loc[common] == cb.loc[common]).all()
    print(f"  Seed {sa} vs {sb}: ARI={ari:.6f}  {'IDENTICAL' if identical else 'DIFFERENT'}")

# Variance summary
aris = []
for sa, sb in combinations(SEEDS, 2):
    common = cluster_results[sa].index.intersection(cluster_results[sb].index)
    aris.append(adjusted_rand_score(cluster_results[sa].loc[common], cluster_results[sb].loc[common]))
print(f"\n  Mean ARI: {np.mean(aris):.4f}")
print(f"  Std ARI:  {np.std(aris):.4f}")
print(f"  Min ARI:  {np.min(aris):.4f}")

/home/sjeong7/.local/lib/python3.13/site-packages/rapids_singlecell/__init__.py:38: UserWarning: 
Multiple rapids_singlecell packages are installed: rapids-singlecell, rapids-singlecell-cu12
Please uninstall all versions and reinstall only one:
  pip uninstall rapids-singlecell rapids-singlecell-cu12

  _detect_duplicate_installation()



--- Seed 0 ---
  9 clusters, 2.2s

--- Seed 1 ---
  8 clusters, 0.3s

--- Seed 42 ---
  9 clusters, 0.3s

Seed robustness: pairwise ARI
  Seed 0 vs 1: ARI=0.873602  DIFFERENT
  Seed 0 vs 42: ARI=0.937118  DIFFERENT
  Seed 1 vs 42: ARI=0.846176  DIFFERENT

  Mean ARI: 0.8856
  Std ARI:  0.0381
  Min ARI:  0.8462


In [2]:
# Module score stability across seeds
adata = sc.read_h5ad(dcfg["canonical_h5ad"])
if "counts" in adata.layers:
    adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata, target_sum=10000)
sc.pp.log1p(adata)
for name, genes in dcfg["known_marker_sets"].items():
    present = [g for g in genes if g in adata.var_names]
    if len(present) >= 2:
        sc.tl.score_genes(adata, gene_list=present, score_name=f"score_{name}", use_raw=False)

score_cols = [c for c in adata.obs.columns if c.startswith("score_")]
scores = adata.obs[score_cols].copy()
scores.index = adata.obs_names.str.replace("-1$", "", regex=True)

from itertools import combinations
for sa, sb in combinations(SEEDS, 2):
    ca, cb = cluster_results[sa], cluster_results[sb]
    common = scores.index.intersection(ca.index).intersection(cb.index)
    
    ct = pd.crosstab(ca.loc[common], cb.loc[common])
    ri, ci = linear_sum_assignment(-ct.values)
    matching = {ct.index[i]: ct.columns[j] for i, j in zip(ri, ci)}
    
    mean_a = scores.loc[common].assign(cl=ca.loc[common].values).groupby("cl").mean()
    mean_b = scores.loc[common].assign(cl=cb.loc[common].values).groupby("cl").mean()
    
    rhos = []
    for a, b in matching.items():
        if a in mean_a.index and b in mean_b.index:
            r = spearmanr(mean_a.loc[a], mean_b.loc[b]).statistic
            rhos.append(r)
    
    print(f"Seed {sa} vs {sb}: ARI={adjusted_rand_score(ca.loc[common], cb.loc[common]):.3f}  Module rho={np.mean(rhos):.3f}")

Seed 0 vs 1: ARI=0.874  Module rho=0.996
Seed 0 vs 42: ARI=0.937  Module rho=1.000
Seed 1 vs 42: ARI=0.846  Module rho=0.996
